In [2]:
%load_ext autoreload
%autoreload 2

In [4]:
from datetime import datetime
import os

import geopandas as gpd
import keras

import gee
import utils

In [6]:
region_name = 'test_region'

tile_size = 576 # this is the around the max size that GEE exports allow with 12-band imagery
tile_padding = 24

start_date = datetime(2023, 1, 1)
end_date = datetime(2023, 12, 31)
clear_threshold = 0.6

pred_threshold = 0.1

In [8]:
# load an ensembled model
model_name = '48px_v3.7-ensemble_2024-01-17'
model = keras.models.load_model(f'../models/{model_name}.h5')
region = gpd.read_file(f'../data/boundaries/{region_name}.geojson').geometry[0].__geo_interface__
tiles = utils.create_tiles(region, tile_size, tile_padding)
print(f"Created {len(tiles):,} tiles")



Created 2,212 tiles


In [10]:
data_pipeline = gee.S2_Data_Extractor(
    tiles, 
    start_date, 
    end_date, 
    clear_threshold, 
    batch_size=500
    )

C:\Users\daxin\anaconda3\Lib\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


EEException: Caller does not have required permission to use project earthindex. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam/project?project=earthindex and then retry. Propagation of the new permission may take a few minutes.

In [ ]:
data_pipeline.make_predictions(model, pred_threshold=pred_threshold)

In [ ]:
# write the predictions to a file
print(len(data_pipeline.predictions), 'chips with predictions above', pred_threshold)
# write the predictions to a file
model_version_name = '_'.join(model_name.split('_')[0:2])
# if the outputs directory does not exist, create it
if not os.path.exists(f'../data/outputs/{model_version_name}'):
    os.makedirs(f'../data/outputs/{model_version_name}')
time_period = f"{start_date.month}_{start_date.year}-{end_date.month}_{end_date.year}"
data_pipeline.predictions.to_file(f"../data/outputs/{model_version_name}/{region_name}_{model_version_name}_{pred_threshold:.2f}_{time_period}.geojson", driver="GeoJSON")